### Instalação das dependências

Os pacotes abaixo são necessários para a otimização de hiperparâmetros com RandomizedSearchCV. Execute esta célula antes de qualquer outra.

In [ ]:
!pip install scikit-learn pandas scipy -q

# Código 2.2 — Otimização de Hiperparâmetros com RandomizedSearchCV

**Objetivo:** Realizar a busca aleatória de hiperparâmetros integrando `RandomizedSearchCV` e `Pipeline` para evitar vazamento de dados.

### Passo 1: Preparação dos Dados e Imports

Importamos as ferramentas necessárias e carregamos o dataset.

In [ ]:
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Carga dos dados
df = pd.read_csv("dados_churn.csv")
X = df.drop(columns=["churn"])
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Passo 2: Definição do Pipeline e Espaço de Busca (*Search Space*)

Note que no dicionário de hiperparâmetros os nomes das chaves usam o prefixo `model__` para referenciar a etapa do pipeline.

In [ ]:
# Definição do pipeline base
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42, n_jobs=-1))
])

# Espaço de busca (distribuições de probabilidade para amostragem)
param_distributions = {
    "model__n_estimators": randint(50, 300),
    "model__max_depth": randint(3, 15),
    "model__min_samples_split": randint(2, 10),
    "model__max_features": ["sqrt", "log2", None],
}

**Entendendo as variáveis do espaço de busca:**
- `model__n_estimators`: `randint(50, 300)` sorteia números inteiros de árvores entre 50 e 300.
- `model__max_depth`: `randint(3, 15)` sorteia a profundidade máxima das árvores para controlar *overfitting*.
- `model__min_samples_split`: `randint(2, 10)` sorteia o número mínimo de amostras necessárias para dividir um nó interno.
- `model__max_features`: Testa diferentes estratégias de seleção de features a cada divisão de nó (`sqrt`, `log2` ou todas).

### Passo 3: Execução da Busca Aleatória (`RandomizedSearchCV`)

Definimos 20 iterações de busca otimizando a métrica ROC-AUC.

In [ ]:
# Configuração do RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    random_state=42,
    n_jobs=-1,
)

# Executar a otimização
search.fit(X_train, y_train)

print("Melhores Hiperparâmetros Encontrados:")
for param, valor in search.best_params_.items():
    print(f"  - {param}: {valor}")

print(f"Melhora no ROC-AUC (Validação): {search.best_score_:.4f}")

**Entendendo os parâmetros do `RandomizedSearchCV`:**
- `estimator=pipeline`: Passa o objeto Pipeline completo garantindo o isolamento das transformações em cada fold.
- `param_distributions`: Dicionário contendo os hiperparâmetros a serem amostrados.
- `n_iter=20`: Executa exatamente 20 combinações aleatórias de hiperparâmetros.
- `scoring="roc_auc"`: Métrica utilizada para selecionar a melhor combinação.
- `cv=5`: Validação cruzada de 5 folds para cada combinação sorteada.

---

## Conclusão da Aula — RandomizedSearchCV com Pipeline

Integrar a busca de hiperparâmetros diretamente ao `Pipeline` garante que as transformações de dados sejam re-calculadas a cada fold da validação cruzada, mantendo o processo estritamente reprodutível e livre de contaminação.